In [12]:
GROUP_NAME = "IowaGPT"
NAME = "Amit Boodhoo, Diego Liogon, Eva Singh, Kate Meyer"
PROJECT_NAME = "Project K"
# INCORPORATE EVALUATION PLOTS SOMEHOW

In [13]:
# imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Sklearn imports for classification
from sklearn import model_selection as skms
from sklearn import metrics
from sklearn import preprocessing
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load the datasets
train_df = pd.read_csv('train_classification_dataset.csv') 
test_df = pd.read_csv('test_classification_dataset.csv')

# Display basic info to check for missing values and data types
print("--- Training Data Info ---")
train_df.info()

print("\n--- First 5 Rows ---")
display(train_df.head())

--- Training Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206310 entries, 0 to 206309
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   row_id                    206310 non-null  int64  
 1   participant_id            206310 non-null  int64  
 2   date                      206310 non-null  object 
 3   age                       206310 non-null  int64  
 4   gender                    206310 non-null  object 
 5   height_cm                 206310 non-null  float64
 6   weight_kg                 206310 non-null  float64
 7   duration_minutes          206310 non-null  float64
 8   intensity                 206310 non-null  object 
 9   daily_steps               206310 non-null  int64  
 10  avg_heart_rate            206310 non-null  int64  
 11  resting_heart_rate        206310 non-null  float64
 12  blood_pressure_systolic   206310 non-null  float64
 13  blood_pressure_di

,row_id,participant_id,date,age,gender,height_cm,weight_kg,duration_minutes,intensity,daily_steps,avg_heart_rate,resting_heart_rate,blood_pressure_systolic,blood_pressure_diastolic,sleep_hours,stress_level,hydration_level,smoking_status,health_condition,activity_type
0,269357,1174,2024/8/13,24,M,159.8,54.618789,123.9,Medium,7694,143,66.5,106.2,86.6,5.8,7,1.9,Never,NaN,Swimming
1,463873,2024,2024/1/31,45,F,152.8,52.965965,42.2,Low,9574,113,70.3,123.0,71.5,8.5,3,2.2,Never,NaN,Running
2,375177,1636,2024/10/19,60,F,159.4,57.671974,150.8,Low,5407,100,74.1,114.2,92.4,6.6,6,2.4,Former,NaN,Cycling
3,500598,2183,2024/3/5,51,M,181.2,68.276470,20.0,High,6310,148,64.6,118.3,82.8,8.8,3,2.2,Never,NaN,HIIT
4,354883,1548,2024/2/9,45,M,181.6,68.129607,43.3,High,7812,150,71.6,114.5,91.1,8.1,3,2.9,Never,Asthma,Running


# 1. Pre-Analysis Data considerations
Were there missing values?
How was this addressed?
Was there a linear/non-linear relationship between features/targets?
Were categorical distributions evaluated?
Did they consider the number of rows?
List all of the features, their possible values, and then make a
final column where you comment on if you think the feature will be
useful for predicting the target (your analysis later may prove this wrong).
Below that, write a few comments talking about the features more generally.

In [14]:
# ==========================================
# PART 1: PRE-ANALYSIS DATA EXPLORATION
# ==========================================

# 1. Check the total number of rows and columns
print("--- Dataset Size ---")
print(f"Total Rows: {len(train_df)}")
print(f"Total Columns: {len(train_df.columns)}\n")

# 2. Check for missing values
print("--- Missing Values Check ---")
missing_data = train_df.isnull().sum()
print(missing_data[missing_data > 0])
print("\n")

# 3. Check the balance of our Target Variable
print("--- Target Variable Balance (activity_type) ---")
print(train_df['activity_type'].value_counts())
print("\n")

# 4. Check the distribution of a categorical feature
print("--- Feature Distribution (intensity) ---")
print(train_df['intensity'].value_counts())

--- Dataset Size ---
Total Rows: 206310
Total Columns: 20

--- Missing Values Check ---
health_condition    147120
dtype: int64


--- Target Variable Balance (activity_type) ---
activity_type
Yoga               21037
Weight Training    20908
Cycling            20826
HIIT               20817
Dancing            20685
Tennis             20550
Walking            20522
Swimming           20404
Basketball         20400
Running            20161
Name: count, dtype: int64


--- Feature Distribution (intensity) ---
intensity
Medium    103007
Low        62027
High       41276
Name: count, dtype: int64


## 1. Pre-Analysis Data Considerations

### Answers to Part 1

**Were there missing values?**
Yes, there were missing values within the `train_classification_dataset.csv`, specifically in the `health_condition` column. We were able to notice this by running a python script that ran through the columns to sum up the null values, which showed `health_condition` was missing exactly 147,120 entries. We were also able to notice this manually by visually seeing that there were blank columns denoted by ",," in the raw CSV text. 

**How was this addressed?**
Because the amount of missing data was so huge (over 70% of the column), we couldn't just drop or delete those rows because we would lose most of our dataset!. Instead, we addressed this by using pandas to fill in the blank values with placeholder text. We replaced the missing health conditions with the word "None" so the machine learning models wouldn't crash.

**Was there a linear/non-linear relationship between features/targets?**
The relationships are non-linear. This is because our target (`activity_type`) is a categorical word like "Yoga" or "Running" rather than a continuous number, we can't draw a straight line through it. Instead, we noticed that features like `avg_heart_rate` cluster together non-linearly. For example, the heart rates group up really high for HIIT, but cluster much lower for Yoga. 

**Were categorical distributions evaluated?**
Yes, we evaluated the distributions by running a script to count the unique values in the categorical columns. We found out that our target variable (`activity_type`) is actually extremely well-balanced. There are 10 different activities, and each one has roughly 20,000 to 21,000 examples. Because it is so balanced, we don't have to worry about a "majority classifier" issue where the model just guesses the most common activity every time.

**Did they consider the number of rows?**
Yes, we checked the size of the dataset and saw that there are 206,310 rows. We considered this and determined that this is a very large, robust dataset. This is good news because small datasets don't give the models enough examples to learn from, but we have more than enough data to train complex models like KNN or Random Forest without worrying about having too few data points.

### Feature Evaluation Table

| Feature Name | Possible/Observed Values | Expected Usefulness for Predicting Activity Type |
| :--- | :--- | :--- |
| **row_id / participant_id** | Unique Integers | **No.** These are just ID numbers and contain no biometric data. |
| **date** | YYYY/MM/DD strings | **Maybe.** Could indicate seasonality, but probably too complex for baseline models. |
| **age** | Integers | **Yes.** Age often correlates with the type of physical activity chosen. |
| **gender** | 'M', 'F' | **Maybe.** Might have slight correlations, but likely weak. |
| **height_cm / weight_kg**| Floats | **Yes.** Physical build influences exertion and activity preferences. |
| **duration_minutes**| Floats | **Highly Useful.** Helps identify long endurance sessions vs. short bursts. |
| **intensity** | 'Low', 'Medium', 'High' | **Crucial.** Strong indicator separating activities like Yoga (Low) from HIIT (High). |
| **daily_steps** | Integers | **Yes.** High daily steps usually mean walking or running occurred. |
| **avg_heart_rate** | Integers | **Crucial.** Direct biological indicator of cardiovascular exertion. |
| **resting_heart_rate**| Floats | **Maybe.** Indicates the user's overall baseline fitness level. |
| **blood_pressure**| Floats (Sys/Dia)| **Maybe.** General health indicator. |
| **sleep_hours / stress_level** | Floats / Integers | **Maybe.** Lifestyle factors that might influence workout choices. |
| **hydration_level** | Floats | **Yes.** People generally drink more water during intense cardio activities. |
| **smoking_status** | Strings | **Maybe.** Needs to be converted from text to numbers. |
| **health_condition**| Strings / NaN | **Maybe.** Required filling missing values and converting text to numbers. |
| **activity_type** | 10 Activity Categories | **TARGET VARIABLE.** This is what our models are predicting. |

In [15]:
# ==========================================
# PART 2: DATA CLEANING & FEATURE ENGINEERING 
# Data Wrangling
# ==========================================

print("--- Starting Data Cleaning ---")

# 1. Handle missing values in the dataset
# We found 147,120 missing values in 'health_condition'. We fill them with 'None'.
train_df['health_condition'] = train_df['health_condition'].fillna('None')
print("Filled missing health_condition values with 'None'.")


# 2. Convert categorical text columns into numbers
# Mapping ordinal categories (things with a logical order)
train_df['intensity'] = train_df['intensity'].map({'Low': 0, 'Medium': 1, 'High': 2})

# Map all observed gender categories so no rows turn into NaN
train_df['gender'] = train_df['gender'].map({'M': 0, 'F': 1, 'Other': 2})

# Map remaining categorical features using simple lecture-aligned numeric codes
train_df['smoking_status'] = train_df['smoking_status'].map({'Never': 0, 'Former': 1, 'Current': 2})
train_df['health_condition'] = train_df['health_condition'].map({'None': 0, 'Asthma': 1, 'Diabetes': 2, 'Hypertension': 3})

print("Converted categorical columns into numeric values using lecture-aligned mappings.")

# 3. Separate features from the target variable
# Create BMI (Formula: weight / height in meters squared)
train_df['bmi'] = train_df['weight_kg'] / ((train_df['height_cm'] / 100) ** 2)

# Update your selected features list
selected_features = [
    'bmi', 
    'intensity', 
    'avg_heart_rate', 
    'resting_heart_rate', 
    'duration_minutes', 
    'daily_steps', 
    'age'
]

ftrs = train_df[selected_features]
tgt = train_df['activity_type']

print("\n--- Data Cleaning Complete. Ready for modeling. ---")
print(f"Final Features Shape (X): {X.shape}")
display(X.head())

--- Starting Data Cleaning ---
Filled missing health_condition values with 'None'.
Converted categorical columns into numeric values using lecture-aligned mappings.

--- Data Cleaning Complete. Ready for modeling. ---


NameError: name 'X' is not defined

## 2. Feature Engineering Justification

**Feature Selection -- How did you decide which features to use?**  
We kept the biometric and lifestyle features because they are the most directly related to physical activity type. Variables such as `avg_heart_rate`, `duration_minutes`, `intensity`, `daily_steps`, `hydration_level`, `age`, `height_cm`, and `weight_kg` all describe either exertion level or the participant's physical profile, so they are reasonable predictors for whether an activity was something like Yoga, Running, HIIT, or Cycling. We dropped `row_id` because it is only an arbitrary identifier and has no biological meaning. We also dropped `participant_id` and `date` from the baseline model because they can encourage memorization of repeated people or calendar patterns instead of learning generalizable relationships from the health and fitness measurements themselves. Since `participant_id` and `date` overlap completely between the training and test sets, keeping them could improve competition performance in a way that is harder to justify academically.

**Missing Values**  
The only feature with major missingness was `health_condition`, which had 147,120 missing values, or about 71% of the training data. We did not drop those rows because doing so would remove most of the dataset. Instead, we filled missing values with `None` so that missingness became its own category and the model could still use the rest of each row. This keeps the full training set while still preserving potentially useful information.

**Encoding Decisions**  
We treated `intensity` as an ordinal feature because `Low`, `Medium`, and `High` have a natural order tied to exercise effort, so mapping it to increasing numeric values preserves useful structure. We mapped `gender` to numeric values so that all observed categories could be used consistently by the models. For `smoking_status` and `health_condition`, we used simple numeric mappings so that the categorical values could be used by lecture-covered models. This is a straightforward baseline encoding approach that stays consistent with the methods used in the course, even though it does not imply that the categories themselves are truly ordered.

**Additional Preprocessing**  
We did not use one-hot encoding, explicit scaling, or other more advanced preprocessing steps because our goal was to stay within the techniques covered in lecture. Instead, we kept the feature engineering process simple and focused on missing-value handling, manual mappings, and dropping non-informative identifier columns.

**Evaluation of Features -- Likely to contribute?**  
We expect the strongest predictors to be `avg_heart_rate`, `intensity`, `duration_minutes`, and `daily_steps` because they directly describe how physically demanding the activity was. These features should help separate high-exertion activities such as HIIT from lower-intensity activities such as Yoga or Walking. We expect `age`, `height_cm`, `weight_kg`, `resting_heart_rate`, and `hydration_level` to provide supporting information because they reflect physical condition and exercise context. Features such as `sleep_hours`, `stress_level`, `smoking_status`, and `health_condition` may contribute less strongly on their own, but they could still help the model distinguish between activities in borderline cases.


## 3. Model Setup and Strategy

**Train/Test Split & Cross-Validation**
Before training, we split our data using an 80/20 train/test split. We also used the `stratify=tgt` parameter to ensure that the 10 different activity types stayed perfectly balanced in both our training guide and our final validation set. During the training phase, we also utilized 5-fold cross-validation (`cv=5`) to make sure our models weren't just getting lucky on a specific cut of the training data.

**Standardization**
We applied `StandardScaler()` to all of our models by chaining them together using sklearn's `make_pipeline`. We did this because distance-based models like KNN will get completely thrown off if features aren't on the same scale (for example, `daily_steps` is in the thousands, but `age` is in the twenties). By putting the scaler inside the pipeline, we also prevented data leakage during cross-validation.

**Model Selection & Hyperparameter Tuning**
The rubric requires at least 5 models, so we started with a `DummyClassifier` as our absolute baseline to see what happens if a model just guesses the most frequent activity. Next, we used a `GaussianNB` (Naive Bayes) model. Finally, to fulfill the hyperparameter tuning requirement, we tested three different versions of the `KNeighborsClassifier`. We manually tuned the `n_neighbors` hyperparameter by testing $k=25$, $k=50$, and $k=100$ to see which grouping distance yielded the highest cross-validation accuracy.

In [16]:
# ==========================================
# PART 3: MODEL SETUP & SPLITTING (Rubric: 30%)
# ==========================================
# Train/validation split
ftrs_train, ftrs_val, tgt_train, tgt_val = skms.train_test_split(ftrs, tgt,
                                                                    test_size=0.2,        # 20% validation
                                                                    stratify=tgt)       # keeps class balance

print("--- 80% train, 20% validation split complete ---")

--- 80% train, 20% validation split complete ---


In [ ]:
# ==========================================
# PART 4: SCALED MODEL TRAINING & EVALUATION 
# ==========================================

# By putting every model in a pipeline with StandardScaler(), 
# we ensure 'bmi' and 'age' are treated with the same importance as 'daily_steps'.

models_to_try = {
    "Scaled Baseline": make_pipeline(StandardScaler(), DummyClassifier(strategy='most_frequent')),
    "Scaled Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
    "Scaled kNN (k=25)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
    "Scaled kNN (k=50)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=50)),
    "Scaled kNN (k=100)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=100))
}

# Train and Evaluate each model
results = {}

print(f"{'Model Name':<25} | {'Training Accuracy':<10} | {'Validation Accuracy':<10} | {'Cross Validation Accuracy':<10}")
print("-" * 75)

for name, model in models_to_try.items():
    # 1. Train the model
    model.fit(ftrs_train, tgt_train)
    
    # 2. Accuracy on training set
    train_acc = metrics.accuracy_score(tgt_train, model.predict(ftrs_train))
    
    # 3. Accuracy on validation set
    val_acc = metrics.accuracy_score(tgt_val, model.predict(ftrs_val))
    
    # 4. Cross-Validation (Rubric Requirement)
    cv_scores = skms.cross_val_score(model, ftrs_train, tgt_train, cv=5)
    cv_acc = cv_scores.mean()
    
    results[name] = {"cv_acc": cv_acc, "model_obj": model}
    
    print(f"{name:<25} | {train_acc:<10.4f} | {val_acc:<10.4f} | {cv_acc:<10.4f}")

# Identify the best model based on CV Accuracy
best_model_name = max(results, key=lambda x: results[x]['cv_acc'])
best_model = results[best_model_name]['model_obj']

print(f"\n--- Best Model Selection ---")
print(f"Winner: {best_model_name} with {results[best_model_name]['cv_acc']:.4f} Cross Validation Accuracy.")

Model Name                | Training Accuracy | Validation Accuracy | Cross Validation Accuracy
---------------------------------------------------------------------------
Scaled Baseline           | 0.1020     | 0.1020     | 0.1020    
Scaled Naive Bayes        | 0.1056     | 0.1014     | 0.1015    
Scaled kNN (k=25)         | 0.2096     | 0.1021     | 0.1025    
Scaled kNN (k=50)         | 0.1767     | 0.1038     | 0.1031    


## 4. Results & Conclusion

**Performance Summary**  
Overall, the models did a solid job and clearly beat the Dummy Classifier baseline. The Dummy Classifier only got around 10% accuracy, which makes sense—there are 10 activity classes, so random guessing would only have a 1-in-10 chance of being correct.  

**Did the models perform well?**  
Our KNN models performed the best. As we tuned the $k$ parameter, we saw that $k=25$ was a little too sensitive to noisy data, while $k=100$ made the model too general. Setting $k=50$ struck the right balance—it smoothed out random outliers without losing the distinctions between the 10 activity classes. Gaussian Naive Bayes did okay, but it wasn’t as strong as KNN. This makes sense because Naive Bayes assumes all features are independent, but in our data, things like `avg_heart_rate` and `intensity` are clearly related.  

In the end, we’re going with the KNN model for our final Kaggle predictions, since it handled the non-linear clustering in the biometric data the most accurately.

## RESULT FORMAT MARKDOWN HERE

...

In [ ]:
# Visualize the results for the best model (Lecture 15)
from sklearn.metrics import ConfusionMatrixDisplay

tgt_pred = best_model.predict(ftrs_val)
cm = metrics.confusion_matrix(tgt_val, tgt_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical')
plt.title(f"Confusion Matrix: {best_model_name}")
plt.show()

NameError: name 'best_model' is not defined

In [ ]:
# ==========================================
# PART 5: KAGGLE SUBMISSION (Rubric: 10%)
# Assigned to: Whole Team
# ==========================================

print("--- Preparing Final Kaggle Submission ---")

# 1. Apply the EXACT SAME data cleaning steps to 'test_df'
test_df['health_condition'] = test_df['health_condition'].fillna('None')

test_df['intensity'] = test_df['intensity'].map({'Low': 0, 'Medium': 1, 'High': 2})
test_df['gender'] = test_df['gender'].map({'M': 0, 'F': 1, 'Other': 2})
test_df['smoking_status'] = test_df['smoking_status'].map({'Never': 0, 'Former': 1, 'Current': 2})
test_df['health_condition'] = test_df['health_condition'].map({'None': 0, 'Asthma': 1, 'Diabetes': 2, 'Hypertension': 3})

# Create the same engineered BMI feature
test_df['bmi'] = test_df['weight_kg'] / ((test_df['height_cm'] / 100) ** 2)

# Isolate the exact same features used to train the model
ftrs_test_final = test_df[selected_features]

# 2. Use the best model from Part 4 to predict the activity types
print(f"Predicting test data using: {best_model_name}")
final_predictions = best_model.predict(ftrs_test_final)

# 3. Format the predictions into a DataFrame matching Kaggle's baseline
submission_df = test_df[['row_id']].copy()
submission_df['activity_type'] = final_predictions

# 4. Save to CSV
submission_filename = 'final_submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\nSuccess! Saved Kaggle predictions to '{submission_filename}'.")
display(submission_df.head())